# Orientation checking in `crack_eval_crossply` / `crack_eval_plus_minus`

This notebook demonstrates that `crack_eval_crossply` and `crack_eval_plus_minus` already **check the specimen's ply orientations** against what they expect, and log a warning for any requested orientation that isn't present on the specimen.

It also shows the more general `crack_eval_by_orientation(specimen)` (no `orientations` argument) — since ply orientation is already stored on each `Ply`, calling it without a target list runs detection on *every* orientation actually present, with no mismatch possible. This is effectively the generic "just run it" entry point.

Paths are resolved relative to the repo root automatically, so this notebook works whether Jupyter's working directory is the repo root or this `notebooks/` folder itself.

In [1]:
import logging
from pathlib import Path

from skimage.io import imread

from deladect.detection.crack_detection import (
    crack_eval_by_orientation,
    crack_eval_crossply,
    crack_eval_plus_minus,
)
from deladect.specimen import Specimen

# The warning is emitted via the standard logging module (deladect.detection.crack_detection
# logger), not warnings.warn — configure logging so it prints in this notebook's output.
logging.basicConfig(level=logging.WARNING)


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root (missing {marker!r}) above {Path.cwd()}")


repo_root = find_repo_root()
data_root = repo_root / "example_images" / "sample-1"
frame_paths = sorted(data_root.glob("*.png"))
frame_paths = [p for p in frame_paths if "experimental" not in p.name]


def load_frames():
    """Real sample-1 frames, reused for whatever orientation a demo ply claims to be.
    Orientation is bookkeeping for crack_eval; it does not affect pixel processing."""
    return [imread(str(path)) for path in frame_paths]


len(frame_paths)

5

## 1. Cross-ply: requesting orientations the specimen doesn't have

`crack_eval_crossply` always asks for `[0, 90]`. Build a specimen with a single ply at `45°` instead — no crack detection needs to run at all, since neither requested orientation exists, so this is fast even without real image content lining up with the label.

In [2]:
mismatched_specimen = Specimen(
    name="mismatched-cross-ply",
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    auto_init_stacks=False,
    results_root=str(repo_root / "results"),
)
mismatched_specimen.add_ply(name="ply_45", orientation_deg=45.0)
mismatched_specimen.path_full_list = [str(p) for p in frame_paths]
mismatched_specimen.image_stack_full = []  # not needed: no orientation matches, so crack_eval never runs

results = crack_eval_crossply(mismatched_specimen)
print("results:", results)

results: {}


## 2. Cross-ply: a partial match

Give the specimen a real `0°` ply plus an unexpected `45°` ply (no `90°`). `crack_eval_crossply` runs detection for `0°` (using the real sample-1 frames) and warns only about the missing `90°`.

In [3]:
partial_specimen = Specimen(
    name="partial-cross-ply",
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    auto_init_stacks=False,
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)
partial_specimen.add_ply(name="ply_0", orientation_deg=0.0, avg_crack_width_px=8.0, min_crack_length_px=20.0)
partial_specimen.add_ply(name="ply_45", orientation_deg=45.0)
partial_specimen.path_full_list = [str(p) for p in frame_paths]
partial_specimen.image_stack_full = load_frames()

partial_results = crack_eval_crossply(partial_specimen)
print("orientations detected:", list(partial_results.keys()))

orientations detected: ['0']


## 3. Same check for `crack_eval_plus_minus`

`Specimen.from_plus_minus(angle_deg=30)` builds a `[+30, -30]` laminate (and, since our earlier change, automatically adds the `"30/-30"` interface between them). Asking `crack_eval_plus_minus` for `theta=45` requests `[+45, -45]`, neither of which exists on this specimen — same warning-and-skip behaviour as the cross-ply case.

In [4]:
pm_specimen = Specimen.from_plus_minus(
    name="plus-minus-30",
    angle_deg=30,
    scale_px_mm=41.03328366,
    path_full=str(data_root),
    sorting_key="_sc",
    image_types=["png"],
    auto_init_stacks=False,
    results_root=str(repo_root / "results"),
)
print("plies:", [(p.name, p.orientation_deg) for p in pm_specimen.plies])
print("interfaces:", [(i.name, i.upper_ply_index, i.lower_ply_index) for i in pm_specimen.interfaces])

pm_specimen.path_full_list = [str(p) for p in frame_paths]
pm_specimen.image_stack_full = []  # requested theta=45 doesn't match [+30, -30]; no detection runs

mismatch_results = crack_eval_plus_minus(pm_specimen, theta=45)
print("results:", mismatch_results)

plies: [('ply_0', 30.0), ('ply_1', -30.0)]
interfaces: [('30/-30', 0, 1)]
results: {}


Requesting the correct `theta=30` matches the specimen's actual plies, so detection runs (no warning) and returns `"30"` / `"-30"` keys:

In [5]:
pm_specimen.image_stack_full = load_frames()
match_results = crack_eval_plus_minus(pm_specimen, theta=30)
print("orientations detected:", list(match_results.keys()))

orientations detected: ['30', '-30']


## 4. The general case: no explicit orientation list needed

`crack_eval_by_orientation(specimen)` — called without `orientations` — groups the specimen's plies by whatever orientation they actually have and runs detection for each group. There's nothing to mismatch: it works directly off the plies you already defined on the specimen, so it behaves like a generic `perform_crack_eval(specimen)` for any layup.

In [6]:
auto_results = crack_eval_by_orientation(pm_specimen)
print("orientations detected:", list(auto_results.keys()))

orientations detected: ['30', '-30']
